## Welcome to Week 6 Day 3: Context Engineering with MCP

People used to talk about Prompt Engineering, but now it's given way to a new Skill "Context Engineering".

Philipp Schmid of Google DeepMind wrote the seminal post about Context Engineering:

https://www.philschmid.de/context-engineering

For this week, we will put Context Engineering into practice, heavily using MCP servers.

1. Long-term memory: a knowledge graph the agent writes to and reads back
2. Web search: fresh information from the live web
3. Agentic RAG: a vector store the agent fills from its own research, then searches
4. Integrations: connecting to a live external service, with a local fallback

In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio, create_static_tool_filter
import os
from pathlib import Path
from datetime import datetime
from IPython.display import Markdown, display

load_dotenv(override=True)

True

### A quick note for Windows

Launching a local MCP server from a notebook hits one rough edge on Windows. The server writes its startup output to stderr, but inside a Windows Jupyter kernel that stream has no real file handle behind it, so the launch fails with an `io.UnsupportedOperation: fileno` error, while Mac and Linux are unaffected.

The fix is to send the server's stderr to the null device, so it always has somewhere real to write. We do that once in the next cell, which lets every cell below use `MCPServerStdio` exactly as the OpenAI Agents SDK documents it. On Mac and Linux it costs nothing beyond keeping the server's startup banner out of the notebook.

In [2]:
# On Windows, a stdio MCP server started from a Jupyter kernel writes to a stderr stream with no
# real file descriptor and crashes with io.UnsupportedOperation: fileno. We send the server's
# stderr to the null device so it always has somewhere real to write, which lets every cell below
# use MCPServerStdio exactly as the OpenAI Agents SDK documents it. Mac and Linux are unaffected.
import functools
import subprocess
import agents.mcp.server

agents.mcp.server.stdio_client = functools.partial(agents.mcp.server.stdio_client, errlog=subprocess.DEVNULL)

## Part 1: Long-term memory

Our first context source is memory. This is the official knowledge graph server: it stores entities, observations about them, and the relationships between them, and keeps them on disk between runs. We point it at `memory/memory.json`, so you can open that file and read the graph it builds.

That gives an agent something it normally lacks: a memory that outlives the conversation. The agent writes facts as it learns them and reads them back later.

https://github.com/modelcontextprotocol/servers/tree/main/src/memory

In [15]:
# 파일이 저장될 디렉터리를 미리 만들어 둔다 (exist_ok=True: 이미 있으면 통과)
memory_path = os.path.abspath("memory/memory.json")
os.makedirs(os.path.dirname(memory_path), exist_ok=True)

In [16]:
memory_path = os.path.abspath("memory/memory.json")
memory_params = {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-memory"], "env": {"MEMORY_FILE_PATH": memory_path}}

async with MCPServerStdio(params=memory_params, client_session_timeout_seconds=60) as server:
    memory_tools = await server.list_tools()

memory_tools

[Tool(name='create_entities', title='Create Entities', description='Create multiple new entities in the knowledge graph', inputSchema={'$schema': 'http://json-schema.org/draft-07/schema#', 'type': 'object', 'properties': {'entities': {'type': 'array', 'items': {'type': 'object', 'properties': {'name': {'type': 'string', 'description': 'The name of the entity'}, 'entityType': {'type': 'string', 'description': 'The type of the entity'}, 'observations': {'type': 'array', 'items': {'type': 'string'}, 'description': 'An array of observation contents associated with the entity'}}, 'required': ['name', 'entityType', 'observations']}}}, 'required': ['entities']}, outputSchema={'$schema': 'http://json-schema.org/draft-07/schema#', 'type': 'object', 'properties': {'entities': {'type': 'array', 'items': {'type': 'object', 'properties': {'name': {'type': 'string', 'description': 'The name of the entity'}, 'entityType': {'type': 'string', 'description': 'The type of the entity'}, 'observations': {'

In [17]:
# 에이전트에게 부여할 시스템 지침 — 엔티티 도구를 영속 메모리로 사용하도록 지시
instructions = "너는 엔티티 도구를 영속적인 메모리로 사용해서 대화에 관한 정보를 저장하고 다시 불러온다."

# 사용자 요청문 — 이름, 직무, 강의 주제(MCP)를 소개하는 내용
request = "제 이름은 Ed입니다. 저는 LLM 엔지니어예요. 저는 놀라운 MCP 프로토콜을 포함한 AI 에이전트 강의를 하고 있습니다. MCP는 에이전트를 도구, 리소스, 프롬프트 템플릿과 연결해 주는 프로토콜이며, AI 에이전트에 다양한 기능을 손쉽게 통합할 수 있게 해줍니다."
model = "gpt-5-mini"

In [18]:
async with MCPServerStdio(params=memory_params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

알겠습니다, Ed님. 정보를 저장해 두었습니다.

In [19]:
# MCPServerStdio: stdio 방식으로 MCP 서버 프로세스를 실행하고 연결하는 컨텍스트 매니저
# params=memory_params: 실행할 서버 명령어/인자 등 접속 설정을 전달
# client_session_timeout_seconds=30: 서버 응답을 기다리는 최대 시간(초). 초과 시 타임아웃 발생
# async with 블록을 벗어나면 서버 연결이 자동으로 정리됨
async with MCPServerStdio(params=memory_params, client_session_timeout_seconds=30) as mcp_server:
    # 에이전트 생성 — name은 식별용 이름, instructions는 시스템 지침,
    # model은 사용할 LLM, mcp_servers에는 위에서 연 MCP 서버를 리스트로 전달
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])

    # trace: 이 블록 안에서 일어나는 LLM 호출과 도구 호출을 "conversation"이라는 이름으로 묶어 추적
    with trace("conversation"):
        # Runner.run: 에이전트를 비동기로 실행. 두 번째 인자가 사용자 입력 메시지
        # await로 실행이 끝날 때까지 대기하고, 결과 객체를 result에 저장
        result = await Runner.run(agent, "제 이름은 Ed입니다. 저에 대해 무엇을 알고 있나요?")

    # result.final_output: 에이전트가 최종적으로 생성한 텍스트
    # Markdown()으로 감싸 display()에 넘겨 노트북에서 마크다운 형식으로 렌더링
    display(Markdown(result.final_output))

지금 알고 있는 정보:
- 이름: Ed
- 직업: LLM 엔지니어
- AI 에이전트 강의(특히 MCP 프로토콜)를 가르침

추가하거나 수정할 내용 있으신가요?

### Check out the trace

https://platform.openai.com/traces

## That's it for today

Four MCP servers, four kinds of context: long-term memory, web search, a knowledge base the agent builds itself, and a live integration. Choosing which sources an agent needs, and wiring them in, is much of what context engineering is in practice.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercises</h2>
            <span style="color:#ff7800;">Browse an MCP marketplace like glama.ai/mcp or smithery.ai and add another context source to this notebook, using one of today's four patterns.
            </span>
        </td>
    </tr>
</table>